In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(
        asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy
    ):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

## Local MCP server

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
            "transport": "stdio",
            "command": "python",
            "args": ["resources/2.1_mcp_server.py"],
        }
    }
)

In [ ]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [ ]:
from langchain.agents import create_agent

if not isinstance(prompt, str):
    raise TypeError("Expected a string system prompt")

agent = create_agent(model="gpt-5-nano", tools=tools, system_prompt=prompt)

In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "6"}}

response = await agent.ainvoke(
    {
        "messages": [
            HumanMessage(content="Tell me about the langchain-mcp-adapters library")
        ]
    },
    config=config,
)

In [ ]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='c003a502-d736-44a2-90c1-fce94004bd0e'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 271, 'total_tokens': 363, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EZQZH5yWvlsvtgdOQZQNVebNidw3A', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a14237-c90c-7342-9c5d-f1e9b2746798-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters'}, 'id': 'call_P3traB7VpOVhDaDYk1jBPToJ', 'type': 'tool_call'}],

## Online MCP

In [10]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uv",
            "args": [
                "run",
                "python",
                "-m",
                "mcp_server_time",
                "--local-timezone=America/New_York",
            ],
        }
    }
)

tools = await client.get_tools()

In [ ]:
from pprint import pprint

for t in tools:
    pprint(t)

StructuredTool(name='get_current_time', description='Get current time in a specific timezone', args_schema={'type': 'object', 'properties': {'timezone': {'type': 'string', 'description': "IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 'America/New_York' as local timezone if no timezone provided by the user."}}, 'required': ['timezone']}, metadata={'title': None, 'readOnlyHint': True, 'destructiveHint': False, 'idempotentHint': True, 'openWorldHint': False}, handle_tool_error=<function _handle_mcp_tool_error at 0x0000026A50680D60>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x0000026A553DE8E0>)
StructuredTool(name='convert_time', description='Convert time between timezones', args_schema={'type': 'object', 'properties': {'source_timezone': {'type': 'string', 'description': "Source IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 'America/New_York' as local timezone if no sour

In [34]:
agent = create_agent(
    model="gpt-5-nano",
    # tools=tools,
)

In [35]:
question = HumanMessage(content="What time is it in California?")

response = await agent.ainvoke({"messages": [question]})

pprint(response)
print()
print(response["messages"][-1].content)

{'messages': [HumanMessage(content='What time is it in California?', additional_kwargs={}, response_metadata={}, id='dc1cba9f-0419-48d0-a599-d5bfda6e0e7d'),
              AIMessage(content='California uses Pacific Time (PT). It’s currently Pacific Daylight Time (PDT), which is UTC-7, since daylight saving time is in effect in October. \n\nIf you want the exact current local time, I don’t have real-time clock access. You can send me the current UTC time (or your own time), and I’ll convert it to PT. For example:\n- 12:00 UTC = 05:00 PDT\n- 17:00 UTC = 10:00 PDT\n\nWould you like me to convert a specific time to Pacific Time?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1470, 'prompt_tokens': 13, 'total_tokens': 1483, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1344, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_t

In [36]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
)
response = await agent.ainvoke({"messages": [question]})

pprint(response)
print()
print(response["messages"][-1].content)

{'messages': [HumanMessage(content='What time is it in California?', additional_kwargs={}, response_metadata={}, id='dc1cba9f-0419-48d0-a599-d5bfda6e0e7d'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 220, 'prompt_tokens': 297, 'total_tokens': 517, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EZQt2GszdzWz4sde1b6pMt8vTfPdS', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1424a-7af9-77d2-9dc5-9b5bdf6f6799-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'America/Los_Angeles'}, 'id': 'call_6Hu90PDTwTW2xfO9BhZTWOlT', 'type': 'tool_call'}], invalid_t